Baseline model using linear regression

In [1]:
import os
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df_raw = pd.read_csv('/Users/hariz/Desktop/TMDB-movie-analysis/extract_movie_features.csv')

df = df_raw.copy()
df.columns

Index(['movie_id', 'imdb_id', 'title', 'original_title', 'status',
       'original_language', 'adult', 'video', 'release_date', 'release_year',
       'release_month', 'release_day_of_week', 'budget', 'revenue',
       'net_profit', 'roi', 'runtime', 'movie_popularity', 'vote_average',
       'vote_count', 'collection_name', 'is_part_of_franchise',
       'primary_genre', 'primary_production_company', 'primary_country',
       'director_name', 'cast', 'overview', 'tagline'],
      dtype='str')

In [8]:
df_feature = df[['movie_id', 'original_language', 'release_year', 'release_month', 'release_day_of_week', 'budget', 'roi', 'revenue', 'runtime', 'is_part_of_franchise', 'primary_genre', 'primary_production_company', 'primary_country', 'director_name', 'cast']].copy()

# Convert raw budget and revenue to millions while keeping them numeric
df_feature['budget_m'] = df_feature['budget'] / 1e6
df_feature['revenue_m'] = df_feature['revenue'] / 1e6
df_feature['log_roi'] = np.log1p(df_feature['roi'])
df_feature['log_budget'] = np.log1p(df_feature['budget'])
df_feature['log_revenue'] = np.log1p(df['revenue'])

df_feature.info()

<class 'pandas.DataFrame'>
RangeIndex: 8040 entries, 0 to 8039
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   movie_id                    8040 non-null   int64  
 1   original_language           8040 non-null   str    
 2   release_year                8040 non-null   float64
 3   release_month               8040 non-null   float64
 4   release_day_of_week         8040 non-null   float64
 5   budget                      8040 non-null   int64  
 6   roi                         8040 non-null   float64
 7   revenue                     8040 non-null   int64  
 8   runtime                     8040 non-null   int64  
 9   is_part_of_franchise        8040 non-null   int64  
 10  primary_genre               8039 non-null   str    
 11  primary_production_company  7983 non-null   str    
 12  primary_country             7972 non-null   str    
 13  director_name               8035 non-null   

In [7]:
df_feature.isnull().sum()

movie_id                      0
original_language             0
release_year                  0
release_month                 0
release_day_of_week           0
budget                        0
roi                           0
revenue                       0
runtime                       0
is_part_of_franchise          0
primary_genre                 0
primary_production_company    0
primary_country               0
director_name                 0
cast                          0
budget_m                      0
revenue_m                     0
log_roi                       0
log_budget                    0
log_revenue                   0
dtype: int64

In [6]:
df_feature.dropna(inplace= True, how='any')


In [ ]:
num_cols = [
    'log_budget', 
    'runtime', 
    'is_part_of_franchise',
    'cast_size',
    'main_lead_score',
    'supporting_lead_score',
    'supporting_cast_score',
    'release_year',
    'release_month'
    'release_day'
]

cat_cols = [
    'primary_genre', 
    'primary_production_company',
    'director_name',
    'primary_production_country',
]




In [ ]:
df.describe()

,movie_id,release_year,release_month,release_day_of_week,budget,revenue,net_profit,roi,runtime,movie_popularity,vote_average,vote_count,is_part_of_franchise
count,8.040000e+03,8040.000000,8040.000000,8040.000000,8.040000e+03,8.040000e+03,8.040000e+03,8040.000000,8040.000000,8040.000000,8040.000000,8040.000000,8040.000000
mean,2.066345e+05,2005.207587,6.936567,4.015050,3.227807e+07,9.374970e+07,6.147164e+07,4.252037,112.687562,8.239726,6.503893,2369.822139,0.273259
std,3.274842e+05,14.151327,3.443638,1.254085,4.309790e+07,1.826120e+08,1.548866e+08,11.710634,24.359273,23.582992,0.907995,3809.951559,0.445660
min,5.000000e+00,1925.000000,1.000000,0.000000,5.017170e+05,9.626200e+04,-1.570000e+08,0.100000,31.000000,0.055000,0.000000,0.000000,0.000000
25%,9.979500e+03,1997.000000,4.000000,3.000000,7.000000e+06,1.129993e+07,0.000000e+00,1.000000,97.000000,3.367000,6.000000,312.000000,0.000000
50%,2.617350e+04,2008.000000,7.000000,4.000000,1.800000e+07,3.114227e+07,1.276267e+07,2.128923,108.000000,5.511000,6.600000,1006.000000,0.000000
75%,3.174708e+05,2016.000000,10.000000,5.000000,3.850000e+07,9.416589e+07,5.890021e+07,4.209437,123.000000,9.105000,7.100000,2645.250000,1.000000
max,1.740913e+06,2027.000000,12.000000,6.000000,6.588000e+08,2.923706e+09,2.686706e+09,668.884953,622.000000,1657.233000,10.000000,40885.000000,1.000000
